# 16_label_fingerprints — fingerprint 파일에 라벨 + 4중 검증

**한 줄 요약:** 04에서 만든 fingerprint 엑셀(4시트)에 **active/inactive 라벨**을 붙이고, 라벨이 **정확한 분자에** 붙었는지 4가지 방법으로 검증한다.
**검증:** ① 지문↔SMILES 재계산 일치 ② 라벨 규칙 감사 ③ 시트 간 라벨 일치 ④ 원본 대조.
**큰 흐름:** ① 준비·라벨규칙 → ② 지문 재계산기 → ③ 시트별 라벨+①②검증 → ④~⑧ 나머지 검증·저장

> **📌 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]**. ③은 그 셀에 **처음 나온** 함수·문법(기초 반복은 *(01에서 설명)*).

### 준비 — 폴더 위치 맞추기
어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')  # data/ 폴더를 찾을 때까지 상위로 (하위 폴더에서 열어도 동작)
print('작업 폴더:', os.getcwd())

🔎 *(01에서 설명)*: `os.chdir('..')`=상위 폴더 이동, `print`=출력.

### 셀 1 — 도구 + 라벨 규칙 함수
라이브러리를 가져오고(콘솔 한글 깨짐 방지 포함), IC50→라벨 규칙 함수를 만든다.

In [ ]:
import sys
try:
    sys.stdout.reconfigure(encoding="utf-8")   # 윈도우 콘솔 cp949 → utf-8
except Exception:
    pass
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

SRC = "data/HSD17B13_fingerprints.xlsx"
MERGED = "data/HSD17B13_IC50_merged.xlsx"
OUT = "data/HSD17B13_fingerprints_labeled.xlsx"
ACTIVE_MAX = 10000.0
SHEETS = ["ECFP4", "MACCS", "RDKit", "AtomPair"]
META = ["canonical_smiles", "ic50_nM", "relation", "sources"]


def make_label(ic50, rel):
    rel = str(rel).strip()
    if pd.isna(ic50):
        return np.nan
    if rel in ("<", "<="):
        return 1 if ic50 <= ACTIVE_MAX else 0
    if rel in (">", ">="):
        return 0
    return 1 if ic50 <= ACTIVE_MAX else 0

🔎 **코드 뜯어보기 (셀 1)**
- `sys.stdout.reconfigure(encoding="utf-8")` : 윈도우 콘솔의 한글/기호 깨짐을 막으려 출력 인코딩을 UTF-8로. `def make_label(...)`=IC50/부등호→1/0 (05에서 설명).

### 셀 2 — 지문 재계산기 준비
검증(①)에서 지문을 다시 계산해 저장값과 비교하려고, 04와 똑같은 생성기를 만든다.

In [ ]:
NBITS = 1024
gen_ecfp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=NBITS)
gen_rdk = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=NBITS)
gen_ap = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=NBITS)


def recompute_fp(name, mol):
    if name == "ECFP4":
        return gen_ecfp.GetFingerprintAsNumPy(mol)
    if name == "RDKit":
        return gen_rdk.GetFingerprintAsNumPy(mol)
    if name == "AtomPair":
        return gen_ap.GetFingerprintAsNumPy(mol)
    if name == "MACCS":
        fp = MACCSkeys.GenMACCSKeys(mol)
        arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr

🔎 **코드 뜯어보기 (셀 2)** *(생성기·maccs_np는 04에서 설명)*
- `def recompute_fp(name, mol):` : 지문 종류 이름을 받아 그에 맞는 지문을 다시 계산(검증①에서 저장값과 비교하려고).

### 셀 3 — 시트별 라벨 삽입 + 검증①② 
4개 시트 각각에 라벨을 넣고, 지문을 SMILES로 재계산해 일치하는지(①)와 라벨 규칙(②)을 확인한다.

In [ ]:
merged = pd.read_excel(MERGED, sheet_name="same_dedup_keepdiff")
merged_pairs = set(zip(merged["canonical_smiles"].astype(str),
                       pd.to_numeric(merged["ic50_nM"], errors="coerce")))

print("=" * 70)
labeled_sheets = {}
label_by_smiles = {}   # 물질별 라벨 집합(모순 측정 탐지용)
sheet_labels = {}      # 시트별 행단위 라벨(행 단위 시트 일치 확인용)
all_ok = True

for sheet in SHEETS:
    df = pd.read_excel(SRC, sheet_name=sheet)
    bit_cols = [c for c in df.columns if c not in META]
    # ---- 라벨 계산 & 삽입 ----
    lab_num = [make_label(v, r) for v, r in zip(df["ic50_nM"], df["relation"])]
    activity = ["active" if x == 1 else ("inactive" if x == 0 else "unlabeled")
                for x in lab_num]
    df.insert(len(META), "potency", [("" if pd.isna(x) else int(x)) for x in lab_num])
    df.insert(len(META) + 1, "activity", activity)
    labeled_sheets[sheet] = df
    sheet_labels[sheet] = [(-9 if pd.isna(x) else int(x)) for x in lab_num]

    # ---- 검증 ① 비트↔SMILES 무결성 (전 행 재계산 비교) ----
    stored = df[bit_cols].to_numpy(dtype=np.int8)
    mism_bits = 0
    n_valid = 0
    for i, smi in enumerate(df["canonical_smiles"]):
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            mism_bits += 1
            continue
        rc = np.asarray(recompute_fp(sheet, mol), dtype=np.int8)
        if rc.shape[0] != stored.shape[1] or not np.array_equal(rc, stored[i]):
            mism_bits += 1
        else:
            n_valid += 1

    # ---- 검증 ② 라벨 규칙 감사 (재계산 일치 — 정의상 일치해야) ----
    # (make_label 결정론적이므로 재적용해 불일치 0 확인)
    audit = sum(1 for (v, r), x in zip(zip(df["ic50_nM"], df["relation"]), lab_num)
                if (make_label(v, r) if not (pd.isna(make_label(v, r))) else -9)
                != (x if not pd.isna(x) else -9))

    n_act = activity.count("active")
    n_ina = activity.count("inactive")
    n_unl = activity.count("unlabeled")
    ok = (mism_bits == 0 and audit == 0)
    all_ok = all_ok and ok
    print(f"[{sheet}] 행 {len(df)} | 비트 {len(bit_cols)} | "
          f"active {n_act} / inactive {n_ina} / unlabeled {n_unl}")
    print(f"    ① 비트↔SMILES 재계산 불일치: {mism_bits}  "
          f"{'✔ 전부 일치' if mism_bits == 0 else '✗ 불일치!'}")
    print(f"    ② 라벨 규칙 재적용 불일치: {audit}  {'✔' if audit == 0 else '✗'}")

    for smi, a in zip(df["canonical_smiles"], activity):
        label_by_smiles.setdefault(str(smi), set()).add(a)

🔎 **코드 뜯어보기 (셀 3)**
- `df.insert(위치, "potency", 값)` : 특정 위치에 **새 열 삽입**.
- `stored = df[bit_cols].to_numpy(dtype=np.int8)` : 저장된 지문 비트를 배열로. `np.array_equal(rc, stored[i])`=재계산 지문과 **완전히 같은지**(①).
- `sheet_labels[sheet] = [...]` : 시트별 라벨을 모아둠(③에서 비교용). `label_by_smiles.setdefault(...).add(...)`=물질별 라벨 모으기(⑤용).

### 셀 4 — 검증③: 행 단위 시트 간 라벨 일치
같은 행이 4개 시트에서 같은 라벨인지 확인한다.

In [ ]:
L = np.array([sheet_labels[s] for s in SHEETS])
row_consistent = int(np.all(L == L[0], axis=0).sum())
row_total = L.shape[1]
print("=" * 70)
print(f"③ 행 단위 4시트 라벨 일치: {row_consistent}/{row_total} "
      f"{'✔ 전부 일치' if row_consistent == row_total else '✗'}")

🔎 **코드 뜯어보기 (셀 4)**
- `L = np.array([sheet_labels[s] for s in SHEETS])` : 4시트 라벨을 한 행렬로. `np.all(L == L[0], axis=0)` : 각 행(열 방향 axis=0)이 **모두 첫 시트와 같은지** → 시트 간 일치(③).

### 셀 5 — 참고: 물질 단위 라벨 충돌 표시
같은 물질이 측정마다 라벨이 갈리는 경우(실측 모순, 오류 아님)를 센다.

In [ ]:
conflict_mol = {s: v for s, v in label_by_smiles.items() if len(v) > 1}
print(f"   [참고] 실측이 상반돼 물질 단위로는 라벨이 갈리는 물질: {len(conflict_mol)}건 "
      f"(오류 아님 — 학습 dedup 시 'active 우선'으로 통합)")
inconsistent = {}   # (호환용) 행 단위 일치가 진짜 판정 기준

🔎 **코드 뜯어보기 (셀 5)**
- `{s: v for s, v in label_by_smiles.items() if len(v) > 1}` : 라벨 종류가 2개 이상인 물질(측정마다 갈림). **오류가 아니라** 실측 자체가 모순인 경우.

### 셀 6 — 검증④: 원본 대조
라벨된 표가 원본(병합 엑셀)의 IC50과 맞는지 대조한다.

In [ ]:
ref_df = labeled_sheets["ECFP4"]
present = sum(1 for s, v in zip(ref_df["canonical_smiles"].astype(str),
                                pd.to_numeric(ref_df["ic50_nM"], errors="coerce"))
              if (s, v) in merged_pairs)
print(f"④ 원본 대조: fingerprint 행 {len(ref_df)}개 중 병합원본에 존재 {present}개 "
      f"{'✔ 전부 일치' if present == len(ref_df) else '일부 불일치'}")

🔎 **코드 뜯어보기 (셀 6)**
- `set(zip(...))` : (SMILES, IC50) 쌍들의 집합. `(s, v) in merged_pairs` : 라벨표의 각 행이 원본에도 있는지 확인(④).

### 셀 7 — 규칙 경계값 스팟체크
IC50 경계 근처 예시로 라벨 규칙이 눈으로도 맞는지 확인한다.

In [ ]:
ex = ref_df[["ic50_nM", "relation", "activity"]].copy()
print("\n[규칙 경계값 스팟체크] (직접 눈으로 규칙 확인)")
for cond, desc in [
    ((ex.relation.astype(str).str.strip() == "=") & (ex.ic50_nM <= 10000), "= & IC50<=10000 → active여야"),
    ((ex.relation.astype(str).str.strip() == "=") & (ex.ic50_nM > 10000), "= & IC50>10000 → inactive여야"),
    (ex.relation.astype(str).str.strip().isin([">", ">="]), "'>' → inactive여야"),
    (ex.relation.astype(str).str.strip().isin(["<", "<="]) & (ex.ic50_nM <= 10000), "'<' & <=10000 → active여야"),
]:
    sub = ex[cond].head(2)
    for _, r in sub.iterrows():
        print(f"    IC50={r.ic50_nM}, rel='{r.relation}' → {r.activity}   ({desc})")

🔎 **코드 뜯어보기 (셀 7)**
- `ex[(조건A) & (조건B)]` : IC50/부등호 경계 조합별로 몇 개 행을 뽑아 라벨이 규칙대로인지 눈으로 확인.

### 셀 8 — 저장 + 종합 판정
라벨된 4시트를 새 엑셀로 저장하고 전체 검증 통과 여부를 출력한다.

In [ ]:
with pd.ExcelWriter(OUT, engine="openpyxl") as w:
    for sheet in SHEETS:
        labeled_sheets[sheet].to_excel(w, sheet_name=sheet, index=False)

print("\n" + "=" * 70)
print(f"저장: {OUT}")
verdict_ok = all_ok and (row_consistent == row_total) and (present == len(ref_df))
print(f"종합 검증: {'✔ 모두 통과 (라벨이 정확한 화합물에 붙음)' if verdict_ok else '✗ 문제 있음 — 위 로그 확인'}")

🔎 **코드 뜯어보기 (셀 8)**
- `with pd.ExcelWriter(OUT) as w: ...to_excel(w, sheet_name=...)` : 라벨된 4시트를 새 엑셀로 저장(원본 열려 있어도 안전). `verdict_ok = all_ok and (row_consistent==row_total) and ...` : 모든 검증 통과 여부를 **and**로 종합.